In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import math
from datetime import date
import warnings 

warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('elvisdenverco2024obweekday_export_odbc.csv')
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,11423,2024-08-14 14:09:00,81,en,2024-08-14 13:58:35,2024-08-14 14:09:00,70.113.126.123,https://denver-co-od.etc-research.com/,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11424,2024-08-14 14:09:26,81,en,2024-08-14 14:09:03,2024-08-14 14:09:26,70.113.126.123,https://denver-co-od.etc-research.com/index.ph...,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11425,2024-08-14 14:24:11,81,en,2024-08-14 14:09:28,2024-08-14 14:24:11,70.113.126.123,https://denver-co-od.etc-research.com/index.ph...,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11427,2024-08-14 14:57:00,81,en,2024-08-14 14:36:36,2024-08-14 14:57:00,50.202.171.186,https://denver-co-od.etc-research.com/,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11634,2024-08-19 06:18:09,81,en,2024-08-19 06:10:17,2024-08-27 01:02:36,172.58.57.226,https://denver-co-od.etc-research.com/,-21600,ANC,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df=df[(df['INTERV_INIT']!='999')]
df = df[(df['HAVE_5_MIN_FOR_SURVECode'] == 1)]

In [4]:

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [5]:
# Interviewer Location Code Starts here
for _,row in df.iterrows():
    for i in range(1,7):
        
        if not pd.isna(row[f'ELVIS_USER_LOC{i}_LAT']) and not pd.isna(row[f'ELVIS_USER_LOC{i}_LONG']):
            df.loc[row.name,'ELVIS_USER_LOC_LAT']=row[f'ELVIS_USER_LOC{i}_LAT']
            df.loc[row.name,'ELVIS_USER_LOC_LONG']=row[f'ELVIS_USER_LOC{i}_LONG']
            break
        else:
            df.loc[row.name,'ELVIS_USER_LOC_LAT']=np.nan
            df.loc[row.name,'ELVIS_USER_LOC_LONG']=np.nan

df['Distances'] = 0
# df['Distances'] = -1

# Get unique INTERV_INIT values
unique_interv_init = df['INTERV_INIT'].unique()
unique_interv_init

def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        return -1
    


# Loop through each unique INTERV_INIT value
for interv in unique_interv_init:
    # Filter rows where INTERV_INIT matches the current unique value
    df_filtered = df[df['INTERV_INIT'] == interv]
    df_filtered.sort_values(by='id',ascending=True,inplace=True)
    
    # If only one row is present, keep the distance as -1
    if len(df_filtered) == 1:
        continue
    
    # Calculate the geodesic distance between consecutive rows
    distances = [0]  # Start with 0 for the first row

    for i in range(1, len(df_filtered)):
        # Get the coordinates for the current and previous row
        coords_1 = (df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LAT'], df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LONG'])
        coords_2 = (df_filtered.iloc[i]['ELVIS_USER_LOC_LAT'], df_filtered.iloc[i]['ELVIS_USER_LOC_LONG'])
        # Calculate distance using geopy
        distance = get_distance_between_coordinates(df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LAT'], df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LONG'],df_filtered.iloc[i]['ELVIS_USER_LOC_LAT'],df_filtered.iloc[i]['ELVIS_USER_LOC_LONG'])
        distances.append(round(distance,4))
    
    # Update the distances in the original DataFrame
    df.loc[df['INTERV_INIT'] == interv, 'Distances'] = distances


for interv in unique_interv_init:
    df_filtered = df[df['INTERV_INIT'] == interv]
    df_filtered.sort_values(by='id', ascending=True, inplace=True)

    distances = df_filtered['Distances'].values
    flags = [0] * len(distances)  # Initialize all flags to 0

    # Loop through the Distances column and check differences
    for i in range(len(distances) - 3):  # Ensure we have at least 4 points
        # Calculate differences between consecutive distances
        diff_1 = abs(distances[i+1] - distances[i])
        diff_2 = abs(distances[i+2] - distances[i+1])
        diff_3 = abs(distances[i+3] - distances[i+2])

        # If all three consecutive differences are less than 0.25, flag all 4 surveys
        if diff_1 < 0.25 and diff_2 < 0.25 and diff_3 < 0.25:
            flags[i] = 1
            flags[i+1] = 1
            flags[i+2] = 1
            flags[i+3] = 1

    # Assign the flags to a new column in the original DataFrame
    df.loc[df['INTERV_INIT'] == interv, 'Consecutive_Flag'] = flags


In [6]:
# Interviewer TimeStamp Code Starts Here

timestamp_columns_check=['completed','reviewscrtime','homeaddtime']
timestamp_columns=check_all_characters_present(df,timestamp_columns_check)
timestamp_columns.sort()

df[timestamp_columns[0]]=pd.to_datetime(df[timestamp_columns[0]], errors='coerce')
df[timestamp_columns[1]]=pd.to_datetime(df[timestamp_columns[1]], errors='coerce')
df[timestamp_columns[2]]=pd.to_datetime(df[timestamp_columns[2]], errors='coerce')

df['Full Survey Time'] = ''
df['Field Reviewed'] = ''
# Calculate minutes for Logistics Time
df['Logistics Time'] = (df[timestamp_columns[0]]- df[timestamp_columns[1]]).dt.total_seconds() / 60
# Calculate minutes for Demographic Time
df['Demographic Time'] = ( df[timestamp_columns[2]] - df[timestamp_columns[1]]).dt.total_seconds() / 60
df['Survey Date'] = df['Completed'].dt.date

# Adding Consecutive_Flag So that I Can Calculate the Location Distance Percentage
timestamp_df=df[['id','INTERV_INIT','Survey Date','Full Survey Time','Logistics Time','Demographic Time','Field Reviewed','Consecutive_Flag']]
timestamp_df['Full Survey Time']=timestamp_df['Logistics Time']+timestamp_df['Demographic Time']

timestamp_df['Full Survey Time Flag'] = (timestamp_df['Full Survey Time'] < 4).astype(int)
timestamp_df['Logistics Time Flag'] = (timestamp_df['Logistics Time'] < 2).astype(int)
timestamp_df['Demographic Time Flag'] = (timestamp_df['Demographic Time'] < 2).astype(int)

In [7]:

# Overall Interviewer TimeStamp Summary code starts Here
unique_interv=df['INTERV_INIT'].unique()

over_all_data=[]
for user in unique_interv:
    total_records=timestamp_df[timestamp_df['INTERV_INIT']==user].shape[0]    
    full_time_survey=timestamp_df[(timestamp_df['INTERV_INIT']==user)&(timestamp_df['Full Survey Time Flag']==1)].shape[0]    
    logistic_time_survey=timestamp_df[(timestamp_df['INTERV_INIT']==user)&(timestamp_df['Logistics Time Flag']==1)].shape[0]    
    demographic_time_survey=timestamp_df[(timestamp_df['INTERV_INIT']==user)&(timestamp_df['Demographic Time Flag']==1)].shape[0]\
    # Calculating distance Flag so have to drop this colum while saving it to file
    location_distance_survey=timestamp_df[(timestamp_df['INTERV_INIT']==user)&(timestamp_df['Consecutive_Flag']==1)].shape[0]

    full_time_survey_percentage = round((full_time_survey / total_records) * 100,2) if total_records != 0 else 0
    logistic_time_survey_percentage = round((logistic_time_survey / total_records) * 100,2) if total_records != 0 else 0
    demographic_time_survey_percentage = round((demographic_time_survey / total_records) * 100,2) if total_records != 0 else 0
    location_distance_survey_percentage = round((location_distance_survey / total_records) * 100,2) if total_records != 0 else 0
    
    # Append results to the list
    over_all_data.append({
        'INTERV_INIT': user,
        'Total Records': total_records,
        'Full Time Survey Records': full_time_survey,
        'Logistic Time Survey Records': logistic_time_survey,
        'Demographic Time Survey Records': demographic_time_survey,
        'Full Time Survey Percentage': full_time_survey_percentage,
        'Logistic Time Survey Percentage': logistic_time_survey_percentage,
        'Demographic Time Survey Percentage': demographic_time_survey_percentage,
        'Location Distance Percentage': location_distance_survey_percentage,
    })

over_all_data_df=pd.DataFrame(over_all_data)

In [8]:
daily_timestamp_data = []


for user in unique_interv:

    user_data = timestamp_df[timestamp_df['INTERV_INIT'] == user]
    

    unique_dates = user_data['Survey Date'].unique()
    

    for date in unique_dates:
        date_data = user_data[user_data['Survey Date'] == date]
        total_records = date_data.shape[0]

        full_time_survey = date_data[date_data['Full Survey Time Flag'] == 1].shape[0]
        logistic_time_survey = date_data[date_data['Logistics Time Flag'] == 1].shape[0]
        demographic_time_survey = date_data[date_data['Demographic Time Flag'] == 1].shape[0]
        
        # Calculate percentages
        full_time_survey_percentage =  round((full_time_survey / total_records) * 100,2) if total_records != 0 else 0
        logistic_time_survey_percentage = round((logistic_time_survey / total_records) * 100,2) if total_records != 0 else 0
        demographic_time_survey_percentage =  round((demographic_time_survey / total_records) * 100,2) if total_records != 0 else 0
        
        # Append results to the list
        daily_timestamp_data.append({
            'INTERV_INIT': user,
            'Survey Date': date,
            'Total Records': total_records,
            'Full Time Survey Records': full_time_survey,
            'Logistic Time Survey Records': logistic_time_survey,
            'Demographic Time Survey Records': demographic_time_survey,
            'Full Time Survey Percentage': full_time_survey_percentage,
            'Logistic Time Survey Percentage': logistic_time_survey_percentage,
            'Demographic Time Survey Percentage': demographic_time_survey_percentage
        })

daily_timestamp_data_df=pd.DataFrame(daily_timestamp_data)

In [9]:
daily_user_location_data = []

for user in unique_interv:
    all_records=0
    flag_records=0
    user_data = df[df['INTERV_INIT'] == user]
    

    unique_dates = user_data['Survey Date'].unique()
    

    for date in unique_dates:
        
        date_data = user_data[user_data['Survey Date'] == date]
        total_records = date_data.shape[0]
        
        consecutive_flag = date_data[date_data['Consecutive_Flag'] == 1].shape[0]
        
        all_records+=total_records
        flag_records+=consecutive_flag
        # Calculate percentages
        consecutive_flag_percentage =  round((consecutive_flag / total_records) * 100,2) if total_records != 0 else 0
        
        # Append results to the list
        daily_user_location_data.append({
            'INTERV_INIT': user,
            'Survey Date': date,
            '# of Surveys': total_records,
            '# of Consecutive_Flag': consecutive_flag,
            'Consective Flags Percentage': consecutive_flag_percentage,

        })
    if all_records and flag_records:
        daily_user_location_data.append({
            'INTERV_INIT': user+' Total',
            'Survey Date': pd.NaT,
            '# of Surveys': all_records,
            '# of Consecutive_Flag': flag_records,
            'Consective Flags Percentage': round((flag_records/all_records)*100,2),

        })

daily_user_location_data_df=pd.DataFrame(daily_user_location_data)

In [10]:
# daily_user_location_data_df.to_csv('daily_user_location.csv',index=False)

In [11]:
daily_time_distance_df=pd.concat([daily_timestamp_data_df,daily_user_location_data_df],axis=1)

In [12]:
daily_time_distance_df.head()

,INTERV_INIT,Survey Date,Total Records,Full Time Survey Records,Logistic Time Survey Records,Demographic Time Survey Records,Full Time Survey Percentage,Logistic Time Survey Percentage,Demographic Time Survey Percentage,INTERV_INIT,Survey Date,# of Surveys,# of Consecutive_Flag,Consective Flags Percentage
0,ANC,2024-08-19,42.0,25.0,1.0,38.0,59.52,2.38,90.48,ANC,2024-08-19,42,4,9.52
1,ANC,2024-08-20,32.0,20.0,1.0,31.0,62.50,3.12,96.88,ANC,2024-08-20,32,0,0.00
2,ANC,2024-08-21,53.0,20.0,0.0,49.0,37.74,0.00,92.45,ANC,2024-08-21,53,0,0.00
3,ANC,2024-08-22,48.0,26.0,1.0,44.0,54.17,2.08,91.67,ANC,2024-08-22,48,9,18.75
4,ANC,2024-08-23,40.0,12.0,2.0,30.0,30.00,5.00,75.00,ANC,2024-08-23,40,0,0.00


In [13]:
daily_timestamp_data_df.columns

Index(['INTERV_INIT', 'Survey Date', 'Total Records',
       'Full Time Survey Records', 'Logistic Time Survey Records',
       'Demographic Time Survey Records', 'Full Time Survey Percentage',
       'Logistic Time Survey Percentage',
       'Demographic Time Survey Percentage'],
      dtype='object')

In [14]:
daily_user_location_data_df.columns

Index(['INTERV_INIT', 'Survey Date', '# of Surveys', '# of Consecutive_Flag',
       'Consective Flags Percentage'],
      dtype='object')

In [15]:
daily_time_distance_df.columns

Index(['INTERV_INIT', 'Survey Date', 'Total Records',
       'Full Time Survey Records', 'Logistic Time Survey Records',
       'Demographic Time Survey Records', 'Full Time Survey Percentage',
       'Logistic Time Survey Percentage', 'Demographic Time Survey Percentage',
       'INTERV_INIT', 'Survey Date', '# of Surveys', '# of Consecutive_Flag',
       'Consective Flags Percentage'],
      dtype='object')

In [16]:
filtered_user_location_data_df = daily_user_location_data_df[~daily_user_location_data_df['INTERV_INIT'].str.contains('Total', case=False, na=False)]

In [17]:
daily_user_location_data_df.shape

(165, 5)

In [18]:
filtered_user_location_data_df.shape

(145, 5)

In [19]:
daily_timestamp_data_df.columns

Index(['INTERV_INIT', 'Survey Date', 'Total Records',
       'Full Time Survey Records', 'Logistic Time Survey Records',
       'Demographic Time Survey Records', 'Full Time Survey Percentage',
       'Logistic Time Survey Percentage',
       'Demographic Time Survey Percentage'],
      dtype='object')

In [20]:
daily_timedistance_df=pd.merge(filtered_user_location_data_df, daily_timestamp_data_df, on=['INTERV_INIT', 'Survey Date'])

In [21]:
daily_timedistance_df.head()

,INTERV_INIT,Survey Date,# of Surveys,# of Consecutive_Flag,Consective Flags Percentage,Total Records,Full Time Survey Records,Logistic Time Survey Records,Demographic Time Survey Records,Full Time Survey Percentage,Logistic Time Survey Percentage,Demographic Time Survey Percentage
0,ANC,2024-08-19,42,4,9.52,42,25,1,38,59.52,2.38,90.48
1,ANC,2024-08-20,32,0,0.00,32,20,1,31,62.50,3.12,96.88
2,ANC,2024-08-21,53,0,0.00,53,20,0,49,37.74,0.00,92.45
3,ANC,2024-08-22,48,9,18.75,48,26,1,44,54.17,2.08,91.67
4,ANC,2024-08-23,40,0,0.00,40,12,2,30,30.00,5.00,75.00


In [22]:
# daily_timedistance_df.to_csv("Daily_time_distance01.csv",index=False)

In [23]:
daily_timedistance_df.columns

Index(['INTERV_INIT', 'Survey Date', '# of Surveys', '# of Consecutive_Flag',
       'Consective Flags Percentage', 'Total Records',
       'Full Time Survey Records', 'Logistic Time Survey Records',
       'Demographic Time Survey Records', 'Full Time Survey Percentage',
       'Logistic Time Survey Percentage',
       'Demographic Time Survey Percentage'],
      dtype='object')

In [24]:
daily_timedistance_df.columns

Index(['INTERV_INIT', 'Survey Date', '# of Surveys', '# of Consecutive_Flag',
       'Consective Flags Percentage', 'Total Records',
       'Full Time Survey Records', 'Logistic Time Survey Records',
       'Demographic Time Survey Records', 'Full Time Survey Percentage',
       'Logistic Time Survey Percentage',
       'Demographic Time Survey Percentage'],
      dtype='object')

In [34]:
user_data=daily_timedistance_df[daily_timedistance_df['INTERV_INIT']=='ANC']

In [36]:
date

datetime.date(2024, 8, 19)

In [37]:
daily_data=user_data[user_data['Survey Date']==date]

In [40]:
daily_data['Full Time Survey Percentage'][0]

59.52

# Code For One Line Summary Where Daily base percentage is greater than 25

In [45]:
unique_interv = daily_timedistance_df['INTERV_INIT'].unique()
summary_data = []

for user in unique_interv:
    user_data = daily_timedistance_df[daily_timedistance_df['INTERV_INIT'] == user]
    total_records = 0
    full_time_flag_records = 0
    logistic_time_flag_records = 0
    demographic_time_flag_records = 0
    consecutive_flag_records = 0
    full_time_counter = 0
    logistic_time_counter = 0
    demographic_time_counter = 0
    consecutive_counter = 0
    full_time_total_percentage=0
    logistic_time_total_percentage=0
    demographic_time_total_percentage=0
    consecutive_flag_total_percentage=0
    
    unique_dates = user_data['Survey Date'].unique()
    total_days = len(unique_dates)
    
    for date in unique_dates:
#         print(date)
        daily_data = user_data[user_data['Survey Date'] == date]
        total_records += daily_data['# of Surveys'].sum()
        
        # Check for full time flag records
        full_time_records = daily_data['Full Time Survey Records'].sum()
        full_time_percentage=daily_data['Full Time Survey Percentage'].sum()
        if full_time_records > 0 and full_time_percentage>25:
            full_time_counter += 1
            full_time_total_percentage+=full_time_percentage
            full_time_flag_records += full_time_records
        
        # Check for logistic time flag records
        logistic_time_records = daily_data['Logistic Time Survey Records'].sum()
        logistic_time_percentage=daily_data['Logistic Time Survey Percentage'].sum()
        if logistic_time_records > 0 and logistic_time_percentage>25:
            logistic_time_total_percentage+=logistic_time_percentage
            logistic_time_counter += 1
            logistic_time_flag_records += logistic_time_records
        
        # Check for demographic time flag records
        demographic_time_records = daily_data['Demographic Time Survey Records'].sum()
        demographic_time_percentage=daily_data['Demographic Time Survey Percentage'].sum()
        if demographic_time_records > 0 and demographic_time_percentage>25:
            demographic_time_total_percentage+=demographic_time_percentage
            demographic_time_flag_records += demographic_time_records
            demographic_time_counter += 1
        
        # Check for consecutive flag records
        consecutive_flags = daily_data['# of Consecutive_Flag'].sum()
        consecutive_percentage=daily_data['Consective Flags Percentage'].sum()
        if consecutive_flags > 0 and consecutive_percentage>25:
            consecutive_flag_total_percentage+=consecutive_percentage
            consecutive_flag_records += consecutive_flags
            consecutive_counter += 1
    
    if total_records > 0:
#         full_time_flag_per=round((full_time_flag_records / total_records) * 100, 2)
#         logistic_time_flag_per=round((logistic_time_flag_records / total_records) * 100, 2)
#         demographic_time_flag_per=round((demographic_time_flag_records / total_records) * 100, 2)
#         consecutive_flag_per=round((consecutive_flag_records / total_records) * 100, 2)
        description = (
            f"Summary for Interviewer {user}:\n"
            f"- Total Records: {total_records}\n"
            f"- {full_time_counter} out of {total_days} days had flags > 25% affecting {round(full_time_total_percentage/full_time_counter,2) if full_time_counter > 0 else 0}% of full survey records completed in under 4 minutes.\n"
            f"- {logistic_time_counter} out of {total_days} days had flags > 25% affecting {round(logistic_time_total_percentage/logistic_time_counter,2) if logistic_time_counter > 0 else 0}% of logistic part records completed in under 2 minutes.\n"
            f"- {demographic_time_counter} out of {total_days} days had flags > 25% affecting {round(demographic_time_total_percentage/demographic_time_counter,2) if demographic_time_counter > 0 else 0}% of demographic part records completed in under 2 minutes.\n"
            f"- {consecutive_counter} out of {total_days} days had flags > 25% affecting {round(consecutive_flag_total_percentage/consecutive_counter,2) if consecutive_counter > 0 else 0}% consecutive flags."
        )
    else:
        description = f"Summary for Interviewer {user}:\n- No records found."
    
    counters = [full_time_counter, logistic_time_counter, demographic_time_counter, consecutive_counter]

    # Count the number of non-zero values
    non_zero_count = sum(1 for count in counters if count > 0)
    
    summary_data.append({
        'INTERV_INIT': user,
        'Total_Flags':non_zero_count,
        'Description': description
    })

summary_data_df = pd.DataFrame(summary_data)


# Code for One Line Summary Where Overall percentage is greater than 25

In [26]:
# unique_interv = daily_timedistance_df['INTERV_INIT'].unique()
# summary_data = []

# for user in unique_interv:
#     user_data = daily_timedistance_df[daily_timedistance_df['INTERV_INIT'] == user]
#     total_records = 0
#     full_time_flag_records = 0
#     logistic_time_flag_records = 0
#     demographic_time_flag_records = 0
#     consecutive_flag_records = 0
#     full_time_counter = 0
#     logistic_time_counter = 0
#     demographic_time_counter = 0
#     consecutive_counter = 0
    
#     unique_dates = user_data['Survey Date'].unique()
#     total_days = len(unique_dates)
    
#     for date in unique_dates:
#         daily_data = user_data[user_data['Survey Date'] == date]
#         total_records += daily_data['# of Surveys'].sum()
        
#         # Check for full time flag records
#         full_time_records = daily_data['Full Time Survey Records'].sum()
#         if full_time_records > 0:
#             full_time_counter += 1
#             full_time_flag_records += full_time_records
        
#         # Check for logistic time flag records
#         logistic_time_records = daily_data['Logistic Time Survey Records'].sum()
#         if logistic_time_records > 0:
#             logistic_time_counter += 1
#             logistic_time_flag_records += logistic_time_records
        
#         # Check for demographic time flag records
#         demographic_time_records = daily_data['Demographic Time Survey Records'].sum()
#         if demographic_time_records > 0:
#             demographic_time_flag_records += demographic_time_records
#             demographic_time_counter += 1
        
#         # Check for consecutive flag records
#         consecutive_flags = daily_data['# of Consecutive_Flag'].sum()
#         if consecutive_flags > 0:
#             consecutive_flag_records += consecutive_flags
#             consecutive_counter += 1
    
#     if total_records > 0:
#         full_time_flag_per=round((full_time_flag_records / total_records) * 100, 2)
#         logistic_time_flag_per=round((logistic_time_flag_records / total_records) * 100, 2)
#         demographic_time_flag_per=round((demographic_time_flag_records / total_records) * 100, 2)
#         consecutive_flag_per=round((consecutive_flag_records / total_records) * 100, 2)
#         # Initialize the description with a summary line
#         description = f"Summary for Interviewer {user}:\n- Total Records: {total_records}\n"

#         # Add full-time flag details
#         if full_time_flag_per > 25:
#             description += f"- {full_time_counter} out of {total_days} days had {full_time_flag_per}% of full survey records completed in under 4 minutes.\n"
#         else:
#             full_time_counter = 0
#             full_time_flag_per = 0
#             description += f"- {full_time_counter} out of {total_days} days had {full_time_flag_per}% of full survey records completed in under 4 minutes.\n"

#         # Add logistic-time flag details
#         if logistic_time_flag_per > 25:
#             description += f"- {logistic_time_counter} out of {total_days} days had {logistic_time_flag_per}% of logistic part records completed in under 2 minutes.\n"
#         else:
#             logistic_time_counter = 0
#             logistic_time_flag_per = 0
#             description += f"- {logistic_time_counter} out of {total_days} days had {logistic_time_flag_per}% of logistic part records completed in under 2 minutes.\n"

#         # Add demographic-time flag details
#         if demographic_time_flag_per > 25:
#             description += f"- {demographic_time_counter} out of {total_days} days had {demographic_time_flag_per}% of demographic part records completed in under 2 minutes.\n"
#         else:
#             demographic_time_counter = 0
#             demographic_time_flag_per = 0
#             description += f"- {demographic_time_counter} out of {total_days} days had {demographic_time_flag_per}% of demographic part records completed in under 2 minutes.\n"

#         # Add consecutive flag details
#         if consecutive_flag_per > 25:
#             description += f"- {consecutive_counter} days had {consecutive_flag_per}% consecutive flags."
#         else:
#             consecutive_counter = 0
#             consecutive_flag_per = 0
#             description += f"- {consecutive_counter} days had {consecutive_flag_per}% consecutive flags."
#     else:
#         description = f"Summary for Interviewer {user}:\n- No records found."
    
#     counters = [full_time_counter, logistic_time_counter, demographic_time_counter, consecutive_counter]

#     # Count the number of non-zero values
#     non_zero_count = sum(1 for count in counters if count > 0)
    
#     summary_data.append({
#         'INTERV_INIT': user,
#         'Total_Flags':non_zero_count,
#         'Description': description
#     })

# summary_data_df = pd.DataFrame(summary_data)
# summary_data_df.to_csv("Daily_time_distance_Summary03.csv",index=False)

In [46]:
summary_data_df.to_csv("Daily_time_distance_Summary04.csv",index=False)

In [29]:
daily_timedistance_df.columns

Index(['INTERV_INIT', 'Survey Date', '# of Surveys', '# of Consecutive_Flag',
       'Consective Flags Percentage', 'Total Records',
       'Full Time Survey Records', 'Logistic Time Survey Records',
       'Demographic Time Survey Records', 'Full Time Survey Percentage',
       'Logistic Time Survey Percentage',
       'Demographic Time Survey Percentage'],
      dtype='object')

In [28]:
ghghghgh

NameError: name 'ghghghgh' is not defined

In [ ]:
counters=[0,0,3,1]
non_zero_count = sum(1 for count in counters if count > 0)
non_zero_count

In [ ]:
daily_timedistance_df['# of Consecutive_Flag'][0].sum()

In [ ]:
daily_timedistance_df['# of Consecutive_Flag'][0]

In [ ]:
non_zero_values=daily_timedistance_df[['# of Consecutive_Flag','Full Time Survey Records','Logistic Time Survey Records','Demographic Time Survey Records']][1:2].iloc[0]

In [ ]:
non_zero_values

In [ ]:
total_flags=sum(non_zero_values != 0)

In [ ]:
total_flags

In [ ]:
daily_timedistance_df.columns

In [ ]:
daily_stats = daily_timedistance_df.groupby(['INTERV_INIT', 'Survey Date']).agg(
    Total_Records=('Total Records', 'sum'),
    Full_Time_Survey_Records_Percent=('Full Time Survey Percentage', 'mean'),
    Logistic_Time_Survey_Records_Percent=('Logistic Time Survey Percentage', 'mean'),
    Demographic_Time_Survey_Records_Percent=('Demographic Time Survey Percentage', 'mean'),
    Consecutive_Flags_Percentage=('Consective Flags Percentage', 'mean')
).reset_index()

In [ ]:
unique_days = daily_stats.groupby('INTERV_INIT')['Survey Date'].nunique().reset_index()
unique_days.columns = ['INTERV_INIT', 'Unique_Days']

In [ ]:
daily_stats = daily_stats.merge(unique_days, on='INTERV_INIT')

In [ ]:
daily_stats['Full_Time_Survey_Records_Percent_OverAll'] = round((daily_stats['Full_Time_Survey_Records_Percent'] * daily_stats['Total_Records']) / daily_stats['Total_Records'].sum() * 100, 2)
daily_stats['Logistic_Time_Survey_Records_Percent_OverAll'] = round((daily_stats['Logistic_Time_Survey_Records_Percent'] * daily_stats['Total_Records']) / daily_stats['Total_Records'].sum() * 100, 2)
daily_stats['Demographic_Time_Survey_Records_Percent_OverAll'] = round((daily_stats['Demographic_Time_Survey_Records_Percent'] * daily_stats['Total_Records']) / daily_stats['Total_Records'].sum() * 100, 2)
daily_stats['Consecutive_Flags_Percentage_OverAll'] = round((daily_stats['Consecutive_Flags_Percentage'] * daily_stats['Total_Records']) / daily_stats['Total_Records'].sum() * 100, 2)


In [ ]:
summary_stats = daily_stats.groupby('INTERV_INIT').apply(lambda group: pd.Series({
    'Total_Records': group['Total_Records'].sum(),
    'Days_Over_4_Min_Full_Time': (group['Full_Time_Survey_Records_Percent'] > 25).sum(),
    'Days_Over_2_Min_Logistic_Time': (group['Logistic_Time_Survey_Records_Percent'] <= 25).sum(), # Corrected check to <=
    'Days_Over_2_Min_Demographic_Time': (group['Demographic_Time_Survey_Records_Percent'] <= 25).sum(), # Corrected check to <=
    'Days_With_Consecutive_Flags': (group['Consecutive_Flags_Percentage'] > 0).sum(),
    'Total_Days': group['Unique_Days'].max(),
    'Full_Time_Survey_Records_Percent_OverAll': round((group['Full_Time_Survey_Records_Percent'] * group['Total_Records']).sum() / group['Total_Records'].sum(), 2),
    'Logistic_Time_Survey_Records_Percent_OverAll': round((group['Logistic_Time_Survey_Records_Percent'] * group['Total_Records']).sum() / group['Total_Records'].sum(), 2),
    'Demographic_Time_Survey_Records_Percent_OverAll': round((group['Demographic_Time_Survey_Records_Percent'] * group['Total_Records']).sum() / group['Total_Records'].sum(), 2),
    'Consecutive_Flags_Percentage_OverAll': round((group['Consecutive_Flags_Percentage'] * group['Total_Records']).sum() / group['Total_Records'].sum(), 2)
})).reset_index()

In [ ]:
def count_flags(row):
    total_flags = 0
    if row['Days_Over_4_Min_Full_Time'] > 0:
        total_flags += 1
    if row['Days_Over_2_Min_Logistic_Time'] > 0:
        total_flags += 1
    if row['Days_Over_2_Min_Demographic_Time'] > 0:
        total_flags += 1
    if row['Days_With_Consecutive_Flags'] > 0:
        total_flags += 1
    return total_flags

In [ ]:
summary_stats['Total_Flags'] = summary_stats.apply(count_flags, axis=1)

# Function to create summary description for each interviewer
def create_summary(row):
    description = (
        f"Summary for Interviewer {row['INTERV_INIT']}:\n"
        f"- Total Records: {row['Total_Records']}\n"
        f"- {row['Days_Over_4_Min_Full_Time']} out of {row['Total_Days']} days had {row['Full_Time_Survey_Records_Percent_OverAll']}% of full survey records completed in under 4 minutes.\n"
        f"- {row['Days_Over_2_Min_Logistic_Time']} out of {row['Total_Days']} days had {row['Logistic_Time_Survey_Records_Percent_OverAll']}% of logistic part records completed in under 2 minutes.\n"
        f"- {row['Days_Over_2_Min_Demographic_Time']} out of {row['Total_Days']} days had {row['Demographic_Time_Survey_Records_Percent_OverAll']}% of demographic part records completed in under 2 minutes.\n"
        f"- {row['Days_With_Consecutive_Flags']} days had {row['Consecutive_Flags_Percentage_OverAll']}% consecutive flags.\n"
        f"- Total Flags: {row['Total_Flags']}"
    )
    return description

In [ ]:
summary_stats['Description'] = summary_stats.apply(create_summary, axis=1)

# Display the summary
summary_stats[['INTERV_INIT','Total_Flags','Description']].to_csv("Daily_time_distance_Summary02.csv",index=False)